![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Similitud y normalización de textos

En este notebook aprenderá a calcular la similitud entre diferentes textos y a normalizarlos usando sklearn y [nltk](https://www.nltk.org/).

Este notebook tiene una licencia de [Creative Commons Attribution-ShareAlike 3.0 Unported License](http://creativecommons.org/licenses/by-sa/3.0/deed.en_US). Un agradecimiento especial para [
Adrien sieg](https://medium.com/@adriensieg/text-similarities-da019229c894)

## Instrucciones Generales

La similitud y normalización de textos son tecnicas del procesamiento de lenguaje natural. Mientras que la similitud permite identificar que tan similares son un par de textos, la normalización permite convertir una palabra en su forma más básica.

Este notebook esta compuesto por dos secciones. En la primera secciónn, usted beberá a obtener la similitud entre dos textos usando diferentes métricas. En la segunda parte, normalizará el texto del set de noticias populares de UCI, eliminando stopwords y haciedo stemming y lematización. Para conocer más detalles de la base, puede ingresar al siguiente [vínculo](https://archive.ics.uci.edu/ml/datasets/online+news+popularity#).
   
Para realizar la actividad, solo siga las indicaciones asociadas a cada celda del notebook.

In [1]:
import sys
print(sys.version)

3.9.12 (main, Apr  5 2022, 01:53:17) 
[Clang 12.0.0 ]


In [2]:
#pip install numpy==2.0.2 tensorflow==2.19.0 scikit-learn==1.6.1 scipy==1.13.1 nltk==3.9.1 pandas==2.2.3

In [3]:
#pip install -r requirements.txt 

In [4]:
import warnings
warnings.filterwarnings('ignore')

## Similitud de texto

### Similitud de Jaccard
La similitud de Jaccard se define como el tamaño de la intersección dividido por el tamaño de la unión de dos conjuntos.

In [5]:
# Definición función de similitud de Jaccard que recibe como parámetros dos textos y retorna su similitud
def jaccard_similarity(query, document):
    # Calculo de la intersección
    intersection = set(query.split()).intersection(set(document.split()))
    # Calculo de la unión
    union = set(query.split()).union(set(document.split()))
    return len(intersection)/len(union)

In [6]:
# Definición de oraciones para calculo de similitud
s1 = "La intelingencia artificial ayuda a resolver los problemas mas complejos"
s2 = "La inteligencia artificial está creciendo rápidamente y esto puede acarrear diferentes problemas"

In [7]:
# Impresión de la similitud de Jaccard entre las dos frases
jaccard_similarity(s1, s2)

0.15789473684210525

### Similitud de coseno

La similitud del coseno calcula la similitud midiendo el coseno del ángulo entre dos vectores.

### Intuición geométrica: ¿qué mide el coseno?

Imagina que cada frase se convierte en una **flecha (vector) en el espacio**. Cada palabra del vocabulario es una dimensión de ese espacio.

```
        "artificial" ↑
                     │   s1 ↗  ← apunta casi igual que s2 (ángulo pequeño)
                     │  ↗ s2
                     │────────────→ "inteligencia"
```

La similitud de coseno mide el **coseno del ángulo** entre esas dos flechas:

| Ángulo entre vectores | cos(θ) | Interpretación |
|---|---|---|
| 0° | **1.0** | Frases idénticas (misma dirección) |
| 45° | **0.7** | Bastante similares |
| 90° | **0.0** | Sin ninguna palabra en común (perpendiculares) |
| >90° | negativo | Imposible con conteos (siempre son ≥ 0) |

**Fórmula matemática:**
$$\text{similitud\_coseno}(v_1, v_2) = \frac{v_1 \cdot v_2}{\|v_1\| \times \|v_2\|}$$

- $v_1 \cdot v_2$ = **producto punto**: suma de los productos elemento a elemento → cuenta palabras compartidas  
- $\|v\|$ = **magnitud**: longitud del vector → normaliza por el tamaño de cada frase

A diferencia de Jaccard (que solo mira si una palabra aparece o no), el coseno también puede capturar **frecuencias** de palabras.

In [8]:
# Importación librerías
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import cosine
import numpy as np

#### Similitud de coseno CountVectorizer
Al vectorizar con CountVectorizer, este tiene la limitación que palabras de un carácter no se consideran dentro del vocabulario, por ejemplo las palabras 'a' e 'y'. Con esto se tiene:

#### ¿Qué hace esta función paso a paso?

```python
vect = CountVectorizer(binary=True)          # binary=True → 1 si aparece, 0 si no (sin contar frecuencia)
X_dtm = vect.fit_transform([s1, s2]).toarray()  # Matriz 2×N: fila 0 = s1, fila 1 = s2
return 1 - cosine(X_dtm[0], X_dtm[1])       # cosine() de scipy = DISTANCIA → restamos de 1 para SIMILITUD
```

> **Ojo importante**: `scipy.spatial.distance.cosine()` devuelve **distancia** (0 = idénticos, 1 = perpendiculares).  
> Por eso hacemos `1 - cosine(...)` para convertirlo en **similitud** (1 = idénticos, 0 = sin relación).

**Limitación**: `CountVectorizer` ignora tokens de 1 carácter (`"y"`, `"a"`, `"o"`). La versión manual en la sección siguiente resuelve esto.

In [9]:
# Definición función de similitud de Coseno que recibe como parámetros dos textos y retorna su similitud
def cosine_distance_countVectorizer(s1, s2):

    # Uso de CountVectorizer para obtener vectores de una frase
    vect = CountVectorizer(binary=True)
    X_dtm = vect.fit_transform([s1, s2]).toarray()

    return 1-cosine(X_dtm[0],X_dtm[1])

In [10]:
# Impresión de la similitud de coseno entre las dos frases definidas anteriormente
cosine_distance_countVectorizer(s1, s2)

0.30151134457776363

#### ¿Por qué necesitamos la versión manual?

`CountVectorizer` usa una expresión regular interna que **requiere al menos 2 caracteres** por token. Palabras como `"y"` (español) o `"a"`, `"I"` (inglés) se descartan silenciosamente.

La versión manual hace exactamente lo mismo pero con `str.split()`, **sin filtrar** palabras cortas. Por eso los resultados difieren ligeramente.

In [11]:
def obtener_vectores(union, s1, s2):

    s1_l = []
    s2_l = []

    for palabra in union:
        if palabra in s1.split():
            s1_l.append(1)
        else:
            s1_l.append(0)

        if palabra in s2.split():
            s2_l.append(1)
        else:
            s2_l.append(0)

    return s1_l, s2_l

# Definición función de similitud de Coseno que recibe como parámetros dos textos y retorna su similitud
def cosine_distance_manual(s1, s2):

    union = list(set(s1.split()).union(set(s2.split())))

    s1_v, s2_v = obtener_vectores(union, s1, s2)

    return 1-cosine(s1_v, s2_v)

In [ ]:
# ============================================================
# DEMO PASO A PASO: ¿Qué hace cosine_distance_manual?
# (corre esta celda DESPUÉS de definir obtener_vectores arriba)
# ============================================================
import numpy as np
import pandas as pd

print("FRASES:")
print(f"  s1: {s1}")
print(f"  s2: {s2}")
print()

# PASO 1: Vocabulario = unión de palabras de ambas frases
palabras_s1 = set(s1.split())
palabras_s2 = set(s2.split())
union = list(palabras_s1 | palabras_s2)

print(f"Solo en s1 ({len(palabras_s1 - palabras_s2)}):   {sorted(palabras_s1 - palabras_s2)}")
print(f"Compartidas ({len(palabras_s1 & palabras_s2)}):  {sorted(palabras_s1 & palabras_s2)}")
print(f"Solo en s2 ({len(palabras_s2 - palabras_s1)}):   {sorted(palabras_s2 - palabras_s1)}")
print(f"Vocabulario total: {len(union)} palabras únicas\n")

# PASO 2: Vectores binarios (1 = aparece, 0 = no aparece)
s1_v, s2_v = obtener_vectores(union, s1, s2)
df_v = pd.DataFrame({'Palabra': union, 'Vector s1': s1_v, 'Vector s2': s2_v}).sort_values('Palabra')
print("Matriz binaria (cada fila es una palabra del vocabulario):")
print(df_v.to_string(index=False))
print()

# PASO 3: Calcular similitud coseno manualmente
v1, v2 = np.array(s1_v, float), np.array(s2_v, float)
prod   = np.dot(v1, v2)          # palabras en común
mag1   = np.linalg.norm(v1)      # √(palabras únicas en s1)
mag2   = np.linalg.norm(v2)      # √(palabras únicas en s2)
sim    = prod / (mag1 * mag2)

print(f"Producto punto  = {int(prod)}  (palabras en común)")
print(f"Magnitud s1     = √{int(mag1**2)} = {mag1:.4f}")
print(f"Magnitud s2     = √{int(mag2**2)} = {mag2:.4f}")
print(f"\nSimilitud = {int(prod)} / ({mag1:.4f} × {mag2:.4f}) = {sim:.5f}")
print(f"✓  cosine_distance_manual(s1, s2) = {cosine_distance_manual(s1, s2):.5f}")

In [12]:
# Impresión de la similitud de coseno entre las dos frases definidas anteriormente
cosine_distance_manual(s1, s2)

0.27386127875258304

#### ¿Por qué los dos métodos dan resultados distintos?

| Método | Resultado | Qué incluye |
|---|---|---|
| `CountVectorizer` | **0.3015** | Ignora `"y"` (1 carácter) |
| Manual | **0.2739** | Incluye `"y"` → vocabulario más grande |

La palabra `"y"` aparece en `s1` pero no en `s2`. Al incluirla:
- El vector de `s1` tiene un `1` extra → su magnitud crece (denominador sube)  
- La similitud baja levemente

**Para el quiz y ejercicios: usa `cosine_distance_manual`**, es más precisa.

La diferencia entre las distancias de coseno se obtiene por la forma de vectorizar, consideren esta segunda para el desarrollo del quiz. Los invitamos a que entiendan con detalle que hace la función manual.

#### ¿Cómo es diferente el Universal Sentence Encoder?

Los métodos anteriores (Jaccard, coseno BoW) solo miran **qué palabras hay**, no su significado ni orden.

El **Universal Sentence Encoder (USE)** de Google es un modelo de deep learning que produce un vector de 512 dimensiones por frase, entrenado para que frases **semánticamente similares** queden cerca aunque usen palabras distintas.

| | Bag of Words | Universal Sentence Encoder |
|---|---|---|
| Representación | Conteo de palabras | Vector de 512 números (embedding) |
| `"perro"` vs `"can"` | Completamente distintos | Muy similares (sinónimos) |
| Captura el orden | No | Sí |
| Velocidad | Muy rápido | Requiere TensorFlow |

> Esta sección requiere `tensorflow` y `tensorflow_hub`. Si no están instalados las celdas siguientes pueden fallar — no bloquea el resto del notebook.

### Codificación de Oraciones y Similitud de Coseno

La codificación de oraciones es una de las representaciones más populares del vocabulario de documentos. Es capaz de capturar el contexto de una palabra en un documento, la similitud semántica y sintáctica, la relación con otras palabras, etc.

Para esta sección del notebook instale la libreria tensorflow y tensorflow_hub (si aun no las ha instalado) con el comando *!pip install tensorflow* y *!pip install tensorflow_hub* respectivamente.

In [ ]:
# Importación librerías
import tensorflow.compat.v1 as tf
tf.disable_eager_execution()
import tensorflow_hub as hub

In [ ]:
## Importación el módulo TF Hub del Universal Sentence Encoder
module_url = "https://tfhub.dev/google/universal-sentence-encoder/4"
embed = hub.load(module_url)

In [ ]:
# Codificación de las frases anteriormente definidas con la libreria tensorflow
with tf.Session() as session:
    session.run([tf.global_variables_initializer(), tf.tables_initializer()])
    sentences_embeddings = session.run(embed([s1, s2]))

In [ ]:
#Impresión de las codificaciones
sentences_embeddings

In [ ]:
# Impresión de la similitud de coseno entre las dos frases definidas anteriormente usando codificación de oraciones
1-cosine(sentences_embeddings[0], sentences_embeddings[1])

## Normalización de textos

In [ ]:
# Importación librerías
import pandas as pd
import numpy as np
import scipy as sp
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics
from nltk.stem.snowball import SnowballStemmer
%matplotlib inline

In [ ]:
# Carga de datos de archivos .csv
df = pd.read_csv('https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/datasets/mashable_texts.csv', index_col=0)
df.head()

In [ ]:
# Separación de variable de interés (y)
y = df.shares
y.describe()

In [ ]:
# Categorización de la variable de interés (y)
y = pd.cut(y, [0, 893, 1200, 2275, 63200], labels=[0, 1, 2, 3])
y.value_counts()

In [ ]:
# Definición de variable de interés en el dataframe
df['y'] = y

In [ ]:
# Definición de variables predictoras
X = df.text

In [ ]:
# Definición de función que recibe un texto vectorizado y calcula el acurracy de un modelo Naive Bayes
def tokenize_test(vect):
    X_dtm = vect.fit_transform(X)
    print('Features: ', X_dtm.shape[1])
    nb = MultinomialNB()
    print(pd.Series(cross_val_score(nb, X_dtm, y, cv=10)).describe())

### Eliminación de stopwords

### ¿Qué son las stopwords y por qué eliminarlas?

Las **stopwords** son palabras gramaticales muy frecuentes que aportan poca información sobre el *tema* de un texto:

- **Artículos**: the, a, an  
- **Preposiciones**: in, on, at, of, for, to  
- **Conjunciones**: and, or, but  
- **Pronombres**: he, she, it, they  

Como aparecen en todos los documentos por igual, no ayudan al modelo a distinguir un artículo de tecnología de uno de política.

**Efecto esperado**: el vocabulario se reduce significativamente → menos columnas en la DTM → modelo más ligero y potencialmente mejor accuracy.

In [ ]:
# Eliminación de stopwords al usar el parámetro 'stop_words' de la función CountVectorizer()
vect_no_stopw = CountVectorizer(stop_words='english')

In [ ]:
# Impresión de stopwords del texto
print(vect_no_stopw.get_stop_words())

In [ ]:
# ¿Cuántas stopwords usa sklearn? Y cómo transforma un texto
stopwords = vect_no_stopw.get_stop_words()
print(f"Total de stopwords (sklearn, inglés): {len(stopwords)}\n")
print("Primeras 40 (alfabético):", sorted(list(stopwords))[:40])

# Ejemplo visual: qué palabras se eliminan
frase_ej = "The quick brown fox jumps over the lazy dog and it runs away"
tok_todos     = frase_ej.lower().split()
tok_filtrados = [t for t in tok_todos if t not in stopwords]
print(f"\nFrase:         {frase_ej}")
print(f"Con stopwords  ({len(tok_todos)} tokens): {tok_todos}")
print(f"Sin stopwords  ({len(tok_filtrados)} tokens): {tok_filtrados}")
print(f"Reducción: {len(tok_todos)-len(tok_filtrados)} palabras ({(1-len(tok_filtrados)/len(tok_todos)):.0%} del texto)")

In [ ]:
# Desempeño del modelo sin considerar stopwords
tokenize_test(vect_no_stopw)

### Stemming

Stemming es un preprocesamiento del texto en el que para cada palabra se obtiene su raíz o en inglés stem.

#### ¿Cómo funciona el Stemming?

El stemming aplica reglas para cortar sufijos y obtener la **raíz** de cada palabra:

| Palabra | Stem | Regla |
|---|---|---|
| `running` | `run` | quita "-ning" |
| `happily` | `happi` | quita "-ly" → puede no ser una palabra real |
| `studies` | `studi` | quita "-es" |
| `dogs` | `dog` | quita "-s" |

**Ventaja**: `"run"`, `"running"`, `"runs"` → todos colapsan a `"run"` → un solo token  
**Desventaja**: el resultado puede no ser una palabra real del diccionario

In [ ]:
# Inicialización de stemmer
stemmer = SnowballStemmer('english')

In [ ]:
# Creación de matrices de documentos usando CountVectorizer a partir de X
vect = CountVectorizer()
vect.fit(X)

In [ ]:
# Definición de lista con vocabulario de la matriz de documentos
words = list(vect.vocabulary_.keys())[:100]

In [ ]:
# Obtención e impresión de los stem de cada palabra de la lista
print([stemmer.stem(word) for word in words])

In [ ]:
# Mostrar lado a lado: palabra original → stem
import pandas as pd
stems_list = [stemmer.stem(w) for w in words]
df_s = pd.DataFrame({'Original': words, 'Stem': stems_list})
df_s['Cambió'] = df_s['Original'] != df_s['Stem']
print(f"Palabras que SÍ cambiaron: {df_s['Cambió'].sum()} de {len(words)}")
print()
print(df_s[df_s['Cambió']].head(20).to_string(index=False))

### Lematización

La lemmatización es un proceso en el que se busca el lema de cada palabra de un texto, siendo un lema la forma base o de diccionario de una palabra.

### Stemming vs Lematización

| | Stemming | Lematización |
|---|---|---|
| **Cómo funciona** | Corta sufijos con reglas fijas | Busca la forma base en diccionario |
| **Resultado** | Raíz (puede no ser palabra real) | Lema (siempre palabra válida) |
| **`"studies"`** | `studi` | `study` |
| **`"better"`** | `better` (no cambia) | `good` (forma base real) |
| **`"wolves"`** | `wolv` | `wolf` |

**El parámetro `pos` es clave:**
- `pos='n'` (sustantivo, default): `"running"` → `"running"` (no lo cambia)
- `pos='v'` (verbo): `"running"` → `"run"` (reconoce que es gerundio)

In [ ]:
# Importación de librerias
from nltk.stem import WordNetLemmatizer
wordnet_lemmatizer = WordNetLemmatizer()
import nltk
nltk.download('wordnet')

In [ ]:
# Obtención e impresión de los lemas de cada palabra de la lista asumiendo que cada palabra es un sustantivo
print([wordnet_lemmatizer.lemmatize(word) for word in words])

In [ ]:
# Obtención e impresión de los lemas de cada palabra de la lista asumiendo que cada palabra es un verbo
print([wordnet_lemmatizer.lemmatize(word, pos='v') for word in words])

In [ ]:
# Comparación lado a lado: original vs stem vs lema(n) vs lema(v)
import pandas as pd
lemas_n = [wordnet_lemmatizer.lemmatize(w, pos='n') for w in words]
lemas_v = [wordnet_lemmatizer.lemmatize(w, pos='v') for w in words]
stems_  = [stemmer.stem(w) for w in words]

df_cmp = pd.DataFrame({'Original': words, 'Stem': stems_, 'Lema(n)': lemas_n, 'Lema(v)': lemas_v})
mask = (df_cmp['Original'] != df_cmp['Stem']) | (df_cmp['Original'] != df_cmp['Lema(n)']) | (df_cmp['Original'] != df_cmp['Lema(v)'])
print("Palabras donde alguna técnica produce cambio:")
print(df_cmp[mask].to_string(index=False))

In [ ]:
# Definición de la función que tenga como parámetro texto y devuelva una lista de lemas
def split_into_lemmas(text):
    text = text.lower()
    words = text.split()
    return [wordnet_lemmatizer.lemmatize(word) for word in words]

In [ ]:
# Creación de matrices de documentos usando CountVectorizer, usando el parámetro 'split_into_lemmas'
vect_lemas = CountVectorizer(analyzer=split_into_lemmas)

In [ ]:
# Desempeño del modelo al lematizar el texto
tokenize_test(vect_lemas)

In [ ]:
# ============================================================
# RESUMEN: comparación de todas las técnicas de normalización
# ============================================================
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score
import pandas as pd

def evaluar(nombre, vect):
    X_dtm = vect.fit_transform(X)
    scores = cross_val_score(MultinomialNB(), X_dtm, y, cv=10)
    return {'Técnica': nombre, 'Nº features': X_dtm.shape[1], 'Accuracy media': round(scores.mean(), 3)}

resultados = [
    evaluar('Sin normalización (baseline)',  CountVectorizer()),
    evaluar('Sin stopwords',                 CountVectorizer(stop_words='english')),
    evaluar('Con lematización',              CountVectorizer(analyzer=split_into_lemmas)),
]
print(pd.DataFrame(resultados).to_string(index=False))
print("\nBaseline aleatorio (4 clases): ~0.250")